In [1]:
import polars as pl
import ast

pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [6]:
file_path = "../data/s01_raw/recipenlg.csv"

df_raw = pl.read_csv(file_path, n_rows=1000)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (1000, 7)


,title,ingredients,directions,link,source,NER
i64,str,str,str,str,str,str
0,"""No-Bake Nut Cookies""","""[""1 c. firmly packed brown sugar"", ""1/2 c. evaporated milk"", ""1/2 tsp. vanilla"", ""1/2 c. broken nuts…","""[""In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine."", ""St…","""www.cookbooks.com/Recipe-Details.aspx?id=44874""","""Gathered""","""[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""butter"", ""bite size shredded rice biscuits""]"""
1,"""Jewell Ball'S Chicken""","""[""1 small jar chipped beef, cut up"", ""4 boned chicken breasts"", ""1 can cream of mushroom soup"", ""1 c…","""[""Place chipped beef on bottom of baking dish."", ""Place chicken on top of beef."", ""Mix soup and crea…","""www.cookbooks.com/Recipe-Details.aspx?id=699419""","""Gathered""","""[""beef"", ""chicken breasts"", ""cream of mushroom soup"", ""sour cream""]"""
2,"""Creamy Corn""","""[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg. cream cheese, cubed"", ""1/3 c. butter, cubed"", ""1/2 t…","""[""In a slow cooker, combine all ingredients. Cover and cook on low for 4 hours or until heated throu…","""www.cookbooks.com/Recipe-Details.aspx?id=10570""","""Gathered""","""[""frozen corn"", ""cream cheese"", ""butter"", ""garlic powder"", ""salt"", ""pepper""]"""
3,"""Chicken Funny""","""[""1 large whole chicken"", ""2 (10 1/2 oz.) cans chicken gravy"", ""1 (10 1/2 oz.) can cream of mushroom…","""[""Boil and debone chicken."", ""Put bite size pieces in average size square casserole dish."", ""Pour gr…","""www.cookbooks.com/Recipe-Details.aspx?id=897570""","""Gathered""","""[""chicken"", ""chicken gravy"", ""cream of mushroom soup"", ""shredded cheese""]"""
4,"""Reeses Cups(Candy) ""","""[""1 c. peanut butter"", ""3/4 c. graham cracker crumbs"", ""1 c. melted butter"", ""1 lb. (3 1/2 c.) powde…","""[""Combine first four ingredients and press in 13 x 9-inch ungreased pan."", ""Melt chocolate chips and…","""www.cookbooks.com/Recipe-Details.aspx?id=659239""","""Gathered""","""[""peanut butter"", ""graham cracker crumbs"", ""butter"", ""powdered sugar"", ""chocolate chips""]"""


In [8]:
df_raw.select(["title", "ingredients", "NER"]).head(10)

title,ingredients,NER
str,str,str
"""No-Bake Nut Cookies""","""[""1 c. firmly packed brown sugar"", ""1/2 c. evaporated milk"", ""1/2 tsp. vanilla"", ""1/2 c. broken nuts…","""[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""butter"", ""bite size shredded rice biscuits""]"""
"""Jewell Ball'S Chicken""","""[""1 small jar chipped beef, cut up"", ""4 boned chicken breasts"", ""1 can cream of mushroom soup"", ""1 c…","""[""beef"", ""chicken breasts"", ""cream of mushroom soup"", ""sour cream""]"""
"""Creamy Corn""","""[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg. cream cheese, cubed"", ""1/3 c. butter, cubed"", ""1/2 t…","""[""frozen corn"", ""cream cheese"", ""butter"", ""garlic powder"", ""salt"", ""pepper""]"""
"""Chicken Funny""","""[""1 large whole chicken"", ""2 (10 1/2 oz.) cans chicken gravy"", ""1 (10 1/2 oz.) can cream of mushroom…","""[""chicken"", ""chicken gravy"", ""cream of mushroom soup"", ""shredded cheese""]"""
"""Reeses Cups(Candy) ""","""[""1 c. peanut butter"", ""3/4 c. graham cracker crumbs"", ""1 c. melted butter"", ""1 lb. (3 1/2 c.) powde…","""[""peanut butter"", ""graham cracker crumbs"", ""butter"", ""powdered sugar"", ""chocolate chips""]"""
"""Cheeseburger Potato Soup""","""[""6 baking potatoes"", ""1 lb. of extra lean ground beef"", ""2/3 c. butter or margarine"", ""6 c. milk"", …","""[""baking potatoes"", ""extra lean ground beef"", ""butter"", ""milk"", ""salt"", ""pepper"", ""Cheddar cheese"", …"
"""Rhubarb Coffee Cake""","""[""1 1/2 c. sugar"", ""1/2 c. butter"", ""1 egg"", ""1 c. buttermilk"", ""2 c. flour"", ""1/2 tsp. salt"", ""1 ts…","""[""sugar"", ""butter"", ""egg"", ""buttermilk"", ""flour"", ""salt"", ""soda"", ""buttermilk"", ""rhubarb"", ""vanilla""…"
"""Scalloped Corn""","""[""1 can cream-style corn"", ""1 can whole kernel corn"", ""1/2 pkg. (approximately 20) saltine crackers,…","""[""cream-style corn"", ""whole kernel corn"", ""crackers"", ""egg"", ""butter"", ""pepper""]"""
"""Nolan'S Pepper Steak""","""[""1 1/2 lb. round steak (1-inch thick), cut into strips"", ""1 can drained tomatoes, cut up (save liqu…","""[""tomatoes"", ""water"", ""onions"", ""Worcestershire sauce"", ""green peppers"", ""oil""]"""


In [9]:
def parse_str_list(s):
    """
    Parse a stringified list, return a list if successful, else return empty list.
    """
    try:
        result = ast.literal_eval(s)
        if isinstance(result, list):
            return result
        else:
            return []
    except Exception:
        return []
      
df_clean = (
  df_raw
  # 1. Safely parse the stringified lists into actual Python lists
  .with_columns([
    pl.col("ingredients").map_elements(parse_str_list, return_dtype=pl.List(pl.Utf8)).alias("raw_ingredients"),
    pl.col("NER").map_elements(parse_str_list, return_dtype=pl.List(pl.Utf8)).alias("ner_ingredients")
  ])
  # 2. Add unified schema columns
  .with_columns([
    pl.lit("recipenlg").alias("source_dataset"),
    pl.lit("en").alias("language"),
    (pl.lit("recipenlg_") + pl.arange(0, df_raw.height).cast(pl.Utf8)).alias("recipe_id")
  ])
  # 3. Select only the columns used in the final schema
  .select([
    "recipe_id",
    "source_dataset",
    "language",
    "raw_ingredients",
    "ner_ingredients", 
  ])
)

df_clean.head()

recipe_id,source_dataset,language,raw_ingredients,ner_ingredients
str,str,str,list[str],list[str]
"""recipenlg_0""","""recipenlg""","""en""","[""1 c. firmly packed brown sugar"", ""1/2 c. evaporated milk"", … ""3 1/2 c. bite size shredded rice biscuits""]","[""brown sugar"", ""milk"", … ""bite size shredded rice biscuits""]"
"""recipenlg_1""","""recipenlg""","""en""","[""1 small jar chipped beef, cut up"", ""4 boned chicken breasts"", … ""1 carton sour cream""]","[""beef"", ""chicken breasts"", … ""sour cream""]"
"""recipenlg_2""","""recipenlg""","""en""","[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg. cream cheese, cubed"", … ""1/4 tsp. pepper""]","[""frozen corn"", ""cream cheese"", … ""pepper""]"
"""recipenlg_3""","""recipenlg""","""en""","[""1 large whole chicken"", ""2 (10 1/2 oz.) cans chicken gravy"", … ""4 oz. shredded cheese""]","[""chicken"", ""chicken gravy"", … ""shredded cheese""]"
"""recipenlg_4""","""recipenlg""","""en""","[""1 c. peanut butter"", ""3/4 c. graham cracker crumbs"", … ""1 large pkg. chocolate chips""]","[""peanut butter"", ""graham cracker crumbs"", … ""chocolate chips""]"


In [11]:
df_exploded = df_clean.select(["recipe_id", "ner_ingredients"]).explode("ner_ingredients")

print(f"Total unique ner ingredients strings in this 1000-recipe chunk: {df_exploded['ner_ingredients'].n_unique()}")
df_exploded.head(10)

Total unique ner ingredients strings in this 1000-recipe chunk: 1122


recipe_id,ner_ingredients
str,str
"""recipenlg_0""","""brown sugar"""
"""recipenlg_0""","""milk"""
"""recipenlg_0""","""vanilla"""
"""recipenlg_0""","""nuts"""
"""recipenlg_0""","""butter"""
"""recipenlg_0""","""bite size shredded rice biscuits"""
"""recipenlg_1""","""beef"""
"""recipenlg_1""","""chicken breasts"""
"""recipenlg_1""","""cream of mushroom soup"""


In [14]:
df = pl.read_parquet("../data/s03_processed/unified_corpus.parquet")

print(f"Total rows: {df.height:,}")
print("\nSchema details:")
print(df.schema)

print("\nSample Recipe:")
print(df.sample(1).to_dicts()[0])

Total rows: 2,231,142

Schema details:
Schema({'recipe_id': String, 'source_dataset': String, 'language': String, 'raw_ingredients': List(String)})

Sample Recipe:
{'recipe_id': 'recipenlg_1193810', 'source_dataset': 'recipenlg', 'language': 'en', 'raw_ingredients': ['shrimp', 'mayonnaise', 'chicken broth', 'lemon rind', 'lemon juice', 'ginger', 'clove garlic', 'sesame oil', 'fresh asparagus', 'hot cooked penne pasta', 'lemon pepper', 'salt']}
